# 📗 vLLM으로 LoRA 어댑터와 양자화 모델 서빙하기

#### **어댑터·양자화 모델 서빙**

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">저장한 <strong style="font-size:1.13em;color:#1f3a5f;">LoRA 어댑터와 양자화 체크포인트를 vLLM API로 제공</strong>하는 방법입니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">배우는 이유:</strong> 분야별 학습 결과를 선택해 사용하고, 모델을 GPU 메모리에 맞게 실행합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">활용처:</strong> 분야별 상담 어댑터를 제공하는 서버와 AWQ·GPTQ 모델 서버입니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/11_merge_vs_adapter.png" width="1050" alt="어댑터 서빙은 베이스 모델 한 벌에 여러 어댑터를 연결합니다.">

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">노무·전월세 어댑터를 <strong style="font-size:1.13em;color:#1f3a5f;">요청의 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">model</code> 이름으로 선택</strong>합니다. 이어서 원본·GPTQ·AWQ 모델의 메모리와 답변을 비교합니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/10_lesson02_goals.png" width="1050" alt="학습 목표: 어댑터 서버 실행, model 선택, 양자화 서빙, 설정과 로그 확인.">

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">학습 목표</strong> (그림의 번호와 같습니다)</p>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">베이스 모델에 LoRA 어댑터 두 개를 연결해 서버를 실행합니다.</li>
<li style="margin:3px 0;">요청의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">model</code> 이름으로 어댑터를 선택합니다.</li>
<li style="margin:3px 0;">원본·GPTQ·AWQ 모델을 같은 API로 사용합니다.</li>
<li style="margin:3px 0;">서버 로그에서 양자화 인식 방식과 메모리 사용량을 확인합니다.</li>
</ol>
</div>

## 1. 실습 준비

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 01의 서버를 RunPod 터미널에서 종료한 뒤 시작합니다. 이 노트북은 <strong style="font-size:1.13em;color:#1f3a5f;">내 PC에서 실행</strong>하며, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>의 공개 주소·API 키를 그대로 사용합니다.</p>
</div>

In [ ]:
import json
import os
import re
from pathlib import Path

import pandas as pd
from openai import OpenAI
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
material_dir = Path(".").resolve()
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)
day70_dir = material_dir.parent / "day70_SFT_지도파인튜닝"
SERVER_URL = os.environ["VLLM_SERVER_URL"].rstrip("/")  # RunPod 공개 HTTPS 주소
VLLM_API_KEY = os.environ["VLLM_API_KEY"]
MAX_TOKENS = 256     # 답변으로 생성할 최대 토큰 수
pd.set_option("display.max_colwidth", None)

In [ ]:
def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(기존 실습과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

In [ ]:
SYSTEM_PROMPT = "너는 노무 상담 챗봇이다. 노동법 조문에 근거해 존댓말로 간결히 답한다."
labor_eval_rows = load_jsonl(day70_dir / "data" / "본실습" / "labor_eval.jsonl")


SYSTEM_PROMPT_LEASE = "너는 전월세 상담 챗봇이다. 주택임대차보호법 조문에 근거해 존댓말로 간결히 답한다."
lease_eval_rows = load_jsonl(day70_dir / "data" / "따라하기" / "lease_eval.jsonl")

In [ ]:
client = OpenAI(base_url=f"{SERVER_URL}/v1", api_key=VLLM_API_KEY)

## 2. LoRA 어댑터 서빙

### 2-1. 베이스 모델과 어댑터 준비

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">어댑터 서빙은 <strong style="font-size:1.13em;color:#1f3a5f;">베이스 모델 한 벌을 공유</strong>하고 요청에서 사용할 어댑터를 고릅니다. 어댑터의 베이스 모델·모델 클래스·층 이름이 학습할 때와 맞아야 합니다.</p>
<p style="margin:6px 0;">먼저 내 PC에서는 <strong style="font-size:1.13em;color:#1f3a5f;">작은 설정 파일만</strong> 읽어 베이스 모델과 rank를 확인합니다. 모델 가중치와 어댑터 가중치는 아래 RunPod 준비 명령으로 받습니다.</p>
</div>

In [ ]:
from huggingface_hub import hf_hub_download

BASE_MODEL = "Qwen/Qwen3.5-0.8B"
adapter_ids = {
    "labor-lora": "muahseol/enkoa-qwen3.5-0.8b-labor-lora",
    "lease-lora": "muahseol/enkoa-qwen3.5-0.8b-lease-lora",
}
adapter_config_rows = []
for adapter_name, adapter_id in adapter_ids.items():
    config_path = hf_hub_download(adapter_id, "adapter_config.json")
    with open(config_path, encoding="utf-8") as f:
        adapter_config = json.load(f)
    adapter_config_rows.append({
        "어댑터": adapter_name, "r": adapter_config["r"],
        "use_dora": adapter_config["use_dora"],
        "베이스 모델": adapter_config["base_model_name_or_path"],
    })
adapter_config_table = pd.DataFrame(adapter_config_rows)
display(adapter_config_table)

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">두 어댑터의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">r</code>은 8이며 베이스 모델이 같습니다. 서버의 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">--max-lora-rank</code>는 사용할 어댑터의 가장 큰 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">r</code> 이상</strong>이어야 하므로 이 실습에서는 <strong style="font-size:1.13em;color:#1f3a5f;">8</strong>로 맞춥니다. DoRA 어댑터는 병합한 모델로 서빙합니다.</p>
<p style="margin:6px 0;">이 어댑터는 텍스트 전용 클래스로 학습했습니다. 서버에서도 같은 층 이름을 사용하도록 <strong style="font-size:1.13em;color:#1f3a5f;">RunPod에 텍스트 전용 베이스 폴더</strong>를 준비합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">save_original_format=False</code>로 저장해야 vLLM이 같은 가중치 이름을 읽습니다. 아래는 <strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널</strong>에서 한 번 실행합니다. 내 PC에서 모델을 불러오는 코드가 아닙니다.</p>
</div>

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실습자료 폴더에서 새 RunPod 터미널</strong>을 열고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">day75_vLLM_추론엔진_서빙</code> 폴더로 이동합니다. 서버 명령은 이 폴더의 RunPod 전용 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uv.lock</code> 환경을 사용합니다. 부록 01에서 자동 생성하여 내 PC의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에 저장한 API 키를 입력하고, 이 터미널에서 서버를 계속 실행합니다.</p>
</div>

```bash
cd day75_vLLM_추론엔진_서빙
mkdir -p output
read -s -p "서버 API 키 입력: " VLLM_API_KEY
export VLLM_API_KEY
```

**RunPod 터미널 · 파일 준비**

```bash
uv run python - <<'PY'
from pathlib import Path
from huggingface_hub import snapshot_download

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)
snapshot_download('muahseol/enkoa-qwen3.5-0.8b-labor-lora', local_dir=output_dir / 'labor_lora_adapter')
snapshot_download('muahseol/enkoa-qwen3.5-0.8b-lease-lora', local_dir=output_dir / 'lease_lora_adapter')

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

base_id = "Qwen/Qwen3.5-0.8B"
text_base_dir = output_dir / "qwen3.5-0.8b-text"
model = AutoModelForCausalLM.from_pretrained(base_id, dtype=torch.bfloat16)
# vLLM이 읽는 텍스트 전용 구조로 저장합니다.
model.save_pretrained(text_base_dir, save_original_format=False)
AutoTokenizer.from_pretrained(base_id).save_pretrained(text_base_dir)
print("서버 파일 준비 완료")
PY
```

### 2-2. 터미널에서 어댑터 서버 실행

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 01과 같은 실습 폴더의 터미널에서 실행합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--lora-modules</code>는 <strong style="font-size:1.13em;color:#1f3a5f;">요청에 쓸 이름과 어댑터 폴더</strong>를 연결합니다. <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">--max-loras 2</code></strong>는 실습에 등록하는 노무·전월세 어댑터 두 종류를 같은 배치에서 함께 처리할 수 있게 합니다.</p>
</div>

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">명령어 설정</strong></p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:6px 0;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">설정</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 8px;">의미</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uv run vllm serve &lt;모델&gt;</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">day75 폴더의 RunPod 전용 uv 환경에서 모델을 불러와 API 서버를 시작합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--served-model-name</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">클라이언트가 요청의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">model</code>에 넣을 이름입니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--dtype</code> 생략</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">공식 기본값 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">auto</code>로 모델 설정에 맞는 자료형을 선택합니다. 예: <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--dtype bfloat16</code>을 지정하면 16비트 BF16으로 계산합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--max-model-len</code> 생략</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">최대 토큰 수를 모델 설정에서 결정합니다. 예: <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--max-model-len 4096</code>을 지정하면 시스템 메시지·질문·답변을 합쳐 최대 4,096토큰을 사용합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--gpu-memory-utilization</code> 생략</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">공식 기본값 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">0.92</code>로 GPU 메모리의 92%를 실행 예산으로 사용합니다. 명시 예: <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--gpu-memory-utilization 0.92</code>.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--reasoning-parser qwen3</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">Qwen이 출력한 생각 부분과 최종 답변을 구분합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--default-chat-template-kwargs</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">enable_thinking=false</code>로 별도의 생각 생성을 끕니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--host 0.0.0.0</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">Pod 외부에서 들어오는 연결도 받습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--port 8000</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">RunPod에서 열어 둔 8000번 포트로 요청을 받습니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--api-key &quot;$VLLM_API_KEY&quot;</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">환경변수에 저장한 키로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1</code> API 요청을 인증합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tee</code> 로그 저장</td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">서버 출력과 오류를 터미널과 로그 파일에 함께 남깁니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--enable-lora</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">요청마다 LoRA 어댑터를 선택할 수 있게 합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--lora-modules 이름=폴더</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">API에서 부를 어댑터 이름과 실제 폴더를 연결합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--max-loras 2</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">실습에 등록하는 어댑터 두 종류를 같은 배치에서 함께 처리할 수 있게 합니다.</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--max-lora-rank 8</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 8px;">앞 단원에서 학습한 어댑터의 r=8에 맞춰 서버의 rank 상한을 정합니다.</td>
</tr>
</tbody>
</table>
</div>

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">GPU 메모리 예산은 <a style="color:#245fa0;text-decoration:underline;" href="https://docs.vllm.ai/en/v0.27.0/configuration/engine_args/">vLLM 0.27.0 공식 기본값</a>인 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">0.92</code>를 사용합니다. 자료형·최대 길이·LoRA 용량은 각 명령과 표에 표시한 실습 설정을 따릅니다. 모델 선택·인증·외부 접속과 Qwen의 생각 모드 설정도 실습 동작에 맞춰 지정합니다.</p>
</div>

**복사해서 실행 · RunPod 터미널**

```bash
uv run vllm serve output/qwen3.5-0.8b-text \
  --served-model-name qwen3.5-0.8b \
  --reasoning-parser qwen3 \
  --default-chat-template-kwargs '{"enable_thinking": false}' \
  --enable-lora \
  --lora-modules labor-lora=output/labor_lora_adapter lease-lora=output/lease_lora_adapter \
  --max-loras 2 \
  --max-lora-rank 8 \
  --host 0.0.0.0 \
  --port 8000 \
  --api-key "$VLLM_API_KEY" 2>&1 | tee output/lora_server.log
```

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">로그에 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Application startup complete</code></strong>가 나오면 요청을 보낼 수 있습니다. RunPod 터미널은 그대로 두고 <strong style="font-size:1.13em;color:#1f3a5f;">내 PC의 노트북</strong>으로 돌아옵니다. 첫 실행은 모델 다운로드와 GPU 계산 준비에 시간이 걸릴 수 있습니다.</p>
</div>

### 2-3. 모델 목록과 어댑터

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">베이스 모델과 두 어댑터가 각각 어떤 이름으로 등록됐는지 확인합니다.</p>
</div>

In [ ]:
model_rows = [model.to_dict() for model in client.models.list()]
model_table = pd.DataFrame(model_rows)[["id", "root", "parent"]]
display(model_table)

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기:</strong> 어댑터의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">parent</code>는 베이스 서버 이름을 가리킵니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">root</code>는 실제 모델·어댑터 폴더입니다.</p>
</div>

### 2-4. model 이름으로 어댑터 선택

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">같은 노무 질문을 <strong style="font-size:1.13em;color:#1f3a5f;">베이스 모델과 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">labor-lora</code></strong>에 보냅니다. 요청의 다른 조건은 같게 두고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">model</code>만 바꿉니다.</p>
</div>

In [ ]:
adapter_rows = []
for row in labor_eval_rows[:3]:
    messages = [{"role":"system","content":SYSTEM_PROMPT}, {"role":"user","content":row["question"]}]
    base_response = client.chat.completions.create(
        model="qwen3.5-0.8b", messages=messages, temperature=0,
        max_completion_tokens=MAX_TOKENS,
    )
    adapter_response = client.chat.completions.create(
        model="labor-lora", messages=messages, temperature=0,
        max_completion_tokens=MAX_TOKENS,
    )
    adapter_rows.append({
        "질문":row["question"], "정답 답변":row["reference_answer"],
        "Base 답변":base_response.choices[0].message.content,
        "labor-lora 답변":adapter_response.choices[0].message.content,
    })
adapter_table = pd.DataFrame(adapter_rows)
display(adapter_table)

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기:</strong> 답이 같거나 다르다는 사실만으로 어댑터 적용 여부를 판정하지 않습니다. 요청의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">model</code>, 모델 목록, 어댑터 설정을 함께 확인합니다.</p>
</div>

### 2-5. 어댑터 서버 종료와 GPU 메모리 반환

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">RunPod에서 서버를 실행한 터미널</strong>로 돌아가 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code></strong>를 누릅니다. 프롬프트가 돌아오면 같은 터미널에서 확인합니다.</p>
</div>

```bash
nvidia-smi
```

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">vLLM 서버 프로세스가 사라지고 GPU 메모리 사용량이 실행 전 수준으로 줄었는지</strong> 확인합니다. 서버 프로세스가 종료되면 모델 가중치와 KV 캐시 메모리도 반환됩니다. 확인을 마친 뒤 다음 단계로 이동합니다.</p>
</div>

## 3. 양자화 모델 서빙

### 3-1. 모델 설정에서 형식 읽기

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">AWQ·GPTQ</strong>는 앞 단원에서 배운 가중치 양자화 방법입니다. 이번에는 이미 양자화된 체크포인트를 실행합니다.</p>
<p style="margin:6px 0;">vLLM은 모델의 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">quantization_config</code></strong>를 읽어 지원되는 형식을 인식합니다. 실제 인식한 형식은 서버 시작 로그의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">quantization=</code>에서 확인합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">GPTQ는 양자화 방법</strong>이며, 로그의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">compressed-tensors</code>는 이 체크포인트의 저장 형식입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">None</code>은 양자화하지 않은 원본이고, AWQ 모델은 이 실습 환경에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">auto_awq</code>로 표시됩니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/13_quantization_config.png" width="1050" alt="vLLM이 체크포인트의 양자화 설정을 읽고 API로 모델을 제공합니다.">

### 3-2. 질문과 결과 저장 공간 준비

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">같은 노무 질문 3개를 원본·GPTQ·AWQ 모델에 보냅니다. 아래 두 셀은 <strong style="font-size:1.13em;color:#1f3a5f;">비교를 시작할 때 한 번만</strong> 실행합니다.</p>
</div>

In [ ]:
quant_models = [{'모델': '원본 bf16', '저장소': 'Qwen/Qwen3.5-4B', '서버 이름': 'qwen3.5-4b'},
 {'모델': 'GPTQ W4A16',
  '저장소': 'RedHatAI/Qwen3.5-4B-quantized.w4a16',
  '서버 이름': 'qwen3.5-4b-gptq'},
 {'모델': 'AWQ 4비트', '저장소': 'QuantTrio/Qwen3.5-4B-AWQ', '서버 이름': 'qwen3.5-4b-awq'}]
quant_results = {}  # 모델 이름별로 저장합니다. 같은 모델을 다시 실행하면 그 결과를 덮어씁니다.

In [ ]:
quant_questions = [row["question"] for row in labor_eval_rows[:3]]
quant_message_lists = []
for question in quant_questions:
    quant_message_lists.append([("system", SYSTEM_PROMPT), ("human", question)])

### 3-3. 모델을 하나씩 실행

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">어댑터 서버 종료와 메모리 반환 확인을 마친 뒤 진행합니다. 아래 명령은 <strong style="font-size:1.13em;color:#1f3a5f;">한 번에 하나만</strong> 실행합니다. 각 모델에서 준비 완료 로그가 나오면 아래의 결과 수집 셀을 실행합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">진행 순서:</strong> 원본 서버 실행·결과 수집 → <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code> → GPTQ 서버 실행·결과 수집 → <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code> → AWQ 서버 실행·결과 수집 → <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code>.</p>
</div>

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--language-model-only</code>는 이미지·음성 입력을 끄고 텍스트 요청만 받는 설정입니다. 나머지 서버 옵션은 앞 실습과 같습니다.</p>
</div>

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">0: 원본 bf16</strong></p>
</div>

**복사해서 실행 · RunPod 터미널**

```bash
uv run vllm serve Qwen/Qwen3.5-4B \
  --served-model-name qwen3.5-4b \
  --language-model-only \
  --reasoning-parser qwen3 \
  --default-chat-template-kwargs '{"enable_thinking": false}' \
  --host 0.0.0.0 \
  --port 8000 \
  --api-key "$VLLM_API_KEY" 2>&1 | tee output/qwen3.5-4b_server.log
```

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">1: GPTQ W4A16</strong></p>
</div>

**복사해서 실행 · RunPod 터미널**

```bash
uv run vllm serve RedHatAI/Qwen3.5-4B-quantized.w4a16 \
  --served-model-name qwen3.5-4b-gptq \
  --language-model-only \
  --reasoning-parser qwen3 \
  --default-chat-template-kwargs '{"enable_thinking": false}' \
  --host 0.0.0.0 \
  --port 8000 \
  --api-key "$VLLM_API_KEY" 2>&1 | tee output/qwen3.5-4b-gptq_server.log
```

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">2: AWQ 4비트</strong></p>
</div>

**복사해서 실행 · RunPod 터미널**

```bash
uv run vllm serve QuantTrio/Qwen3.5-4B-AWQ \
  --served-model-name qwen3.5-4b-awq \
  --language-model-only \
  --reasoning-parser qwen3 \
  --default-chat-template-kwargs '{"enable_thinking": false}' \
  --host 0.0.0.0 \
  --port 8000 \
  --api-key "$VLLM_API_KEY" 2>&1 | tee output/qwen3.5-4b-awq_server.log
```

### 3-4. 현재 모델의 로그와 답변 수집

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">RunPod에서 현재 모델의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/&lt;서버 이름&gt;_server.log</code>를 내려받아 <strong style="font-size:1.13em;color:#1f3a5f;">내 PC의 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">output</code> 폴더</strong>에 같은 이름으로 저장합니다.</p>
<p style="margin:6px 0;">터미널에서 실행한 모델의 번호로 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">model_index</code>를 바꿉니다</strong>. 원본은 0, GPTQ는 1, AWQ는 2입니다. 다음 세 셀을 모델마다 실행하며, 같은 모델을 다시 실행하면 이전 결과를 덮어씁니다.</p>
</div>

In [ ]:
model_index = 0  # 현재 실행 중인 모델: 0 원본 / 1 GPTQ / 2 AWQ
quant_model = quant_models[model_index]

In [ ]:
log_name = f"{quant_model['서버 이름']}_server.log"
log_text = (output_dir / log_name).read_text(encoding="utf-8")
weight_gib = float(re.search(r"Model loading took ([\d.]+) GiB", log_text).group(1))
kv_cache_tokens = int(re.search(r"GPU KV cache size: ([\d,]+) tokens", log_text).group(1).replace(",", ""))
quant_method = re.search(r"quantization=([\w-]+)", log_text).group(1)

model_info = {"모델":quant_model["모델"], "양자화 방식":quant_method,
              "가중치 메모리(GiB)":weight_gib, "KV 캐시(토큰)":kv_cache_tokens}
display(model_info)

In [ ]:
quant_llm = ChatOpenAI(
    model=quant_model["서버 이름"],
    base_url=f"{SERVER_URL}/v1",
    api_key=VLLM_API_KEY,
    temperature=0,
    max_tokens=MAX_TOKENS,
)
quant_answers = quant_llm.batch(quant_message_lists, config={"max_concurrency": 3})
quant_results[quant_model["모델"]] = {
    "모델": quant_model["모델"],
    "양자화 방식": quant_method,
    "가중치 메모리(GiB)": weight_gib,
    "KV 캐시(토큰)": kv_cache_tokens,
    "답변": [answer.content for answer in quant_answers],
}
display(quant_results[quant_model["모델"]])

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">현재 모델의 수집이 끝났다면 · 원본/GPTQ/AWQ 공통</strong></p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">RunPod에서 서버를 실행한 터미널</strong>로 돌아가 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code></strong>를 누릅니다. 프롬프트가 돌아오면 같은 터미널에서 확인합니다.</p>
</div>

```bash
nvidia-smi
```

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">vLLM 서버 프로세스가 사라지고 GPU 메모리 사용량이 실행 전 수준으로 줄었는지</strong> 확인합니다. 서버 프로세스가 종료되면 모델 가중치와 KV 캐시 메모리도 반환됩니다. 확인을 마친 뒤 다음 단계로 이동합니다.</p>
<p style="margin:6px 0;">다음 모델의 서버 명령으로 돌아가 실행한 뒤, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">model_index</code>를 바꾸고 3-4의 세 셀만 다시 실행합니다. 세 모델을 모두 마쳤다면 3-5로 이동합니다.</p>
</div>

### 3-5. 메모리와 답변 비교

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">세 모델의 결과를 모두 모은 뒤 실행합니다. 가중치가 줄면 캐시에 쓸 여유가 생기지만, <strong style="font-size:1.13em;color:#1f3a5f;">실제 KV 캐시 용량은 모델 구조와 엔진 설정에도 영향을 받습니다</strong>.</p>
</div>

In [ ]:
quant_table = pd.DataFrame(quant_results.values())[
    ["모델", "양자화 방식", "가중치 메모리(GiB)", "KV 캐시(토큰)"]
]
display(quant_table.round(2))

In [ ]:
quant_answer_table = pd.DataFrame({
    "질문": quant_questions,
    "정답 답변": [row["reference_answer"] for row in labor_eval_rows[:3]],
    "원본 답변": quant_results["원본 bf16"]["답변"],
    "GPTQ 답변": quant_results["GPTQ W4A16"]["답변"],
    "AWQ 답변": quant_results["AWQ 4비트"]["답변"],
})
display(quant_answer_table)

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">결과 읽기:</strong> 메모리 사용량과 답변을 함께 봅니다. 문장이 달라졌다는 사실만으로 정답 여부나 품질을 판단하지 않습니다.</p>
</div>

## 4. 🖐️ 함께 따라하기: 전월세 어댑터 선택

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">같은 베이스 모델에서 <strong style="font-size:1.13em;color:#1f3a5f;">전월세 어댑터를 선택</strong>해 상담 답변을 받습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">본실습과 다른 점</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">선택:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">labor-lora</code> 대신 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease-lora</code>를 호출합니다. 베이스와 어댑터를 다시 학습하지 않습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">데이터:</strong> 전월세 시스템 메시지와 평가 질문 앞 3개를 사용합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">비교:</strong> 같은 질문을 베이스와 전월세 어댑터에 보내고 정답 답변과 나란히 봅니다.</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실습 진행 방법:</strong> 주석에 맞춰 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">...</code>를 채웁니다. 완성 코드는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">정답/교안_02_정답.ipynb</code>의 같은 절에 있습니다.</p>
</div>

### 4-1. 어댑터 서버 다시 실행

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">세 모델의 결과 수집과 종료 확인을 마쳤다면 같은 터미널에 다음 명령을 입력합니다.</p>
</div>

**복사해서 실행 · RunPod 터미널**

```bash
uv run vllm serve output/qwen3.5-0.8b-text \
  --served-model-name qwen3.5-0.8b \
  --reasoning-parser qwen3 \
  --default-chat-template-kwargs '{"enable_thinking": false}' \
  --enable-lora \
  --lora-modules labor-lora=output/labor_lora_adapter lease-lora=output/lease_lora_adapter \
  --max-loras 2 \
  --max-lora-rank 8 \
  --host 0.0.0.0 \
  --port 8000 \
  --api-key "$VLLM_API_KEY" 2>&1 | tee output/lora_server.log
```

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">로그에 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Application startup complete</code></strong>가 나오면 요청을 보낼 수 있습니다. RunPod 터미널은 그대로 두고 <strong style="font-size:1.13em;color:#1f3a5f;">내 PC의 노트북</strong>으로 돌아옵니다. 첫 실행은 모델 다운로드와 GPU 계산 준비에 시간이 걸릴 수 있습니다.</p>
</div>

In [ ]:
lease_model_table = pd.DataFrame([model.to_dict() for model in client.models.list()])[["id", "parent"]]
display(lease_model_table)

### 4-2. model 이름으로 전월세 어댑터 선택

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">작성할 것:</strong> (1) 베이스 모델 요청, (2) 전월세 어댑터 요청을 작성합니다. 두 요청 모두 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease_messages</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">temperature=0</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_completion_tokens=MAX_TOKENS</code>를 사용합니다.</p>
</div>

<details><summary>힌트</summary>

```text
접근방법: 교안 02의 2-4를 참고합니다.
모델 목록의 id로 베이스와 어댑터를 구분합니다. 같은 메시지를 보내고 model만 바꿉니다.
```

</details>

In [ ]:
lease_rows = []
for row in lease_eval_rows[:3]:
    lease_messages = [{"role":"system", "content":SYSTEM_PROMPT_LEASE},
                      {"role":"user", "content":row["question"]}]
    # TODO (1): 같은 질문을 베이스 모델에 보냅니다.
    lease_base_response = ...
    # TODO (2): 같은 질문을 전월세 어댑터에 보냅니다.
    lease_response = ...
    lease_rows.append({
        "질문": row["question"], "정답 답변": row["reference_answer"],
        "Base 답변": lease_base_response.choices[0].message.content,
        "lease-lora 답변": lease_response.choices[0].message.content,
    })
lease_comparison = pd.DataFrame(lease_rows)
display(lease_comparison)

### 4-3. 결과 확인과 서버 종료

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">세 질문에 대해 두 모델의 답변이 모두 있는지 확인합니다. <strong style="font-size:1.13em;color:#1f3a5f;">서버의 베이스는 그대로이고 요청의 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">model</code>만 달라졌습니다.</strong> 답변이 같다는 이유만으로 어댑터가 적용되지 않았다고 판단하지 않습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">RunPod에서 서버를 실행한 터미널</strong>로 돌아가 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code></strong>를 누릅니다. 프롬프트가 돌아오면 같은 터미널에서 확인합니다.</p>
</div>

```bash
nvidia-smi
```

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">vLLM 서버 프로세스가 사라지고 GPU 메모리 사용량이 실행 전 수준으로 줄었는지</strong> 확인합니다. 서버 프로세스가 종료되면 모델 가중치와 KV 캐시 메모리도 반환됩니다. 확인을 마친 뒤 다음 단계로 이동합니다.</p>
</div>

## 5. 핵심 정리

<div class="day75-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">어댑터 서버:</strong> 베이스 모델 한 벌을 공유하고 요청의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">model</code>로 어댑터를 선택합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">양자화 서버:</strong> AWQ·GPTQ 설정을 읽은 모델을 같은 API로 제공합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">모델 전환:</strong> 터미널에서 앞 서버를 종료한 뒤 다음 서버를 실행합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">비교:</strong> 같은 질문과 생성 조건으로 메모리·답변을 확인합니다.</li>
</ul>
</div>